# [01] CAS(ME)² — Spotting

Menjalankan algoritma spotting pada 33 video `rawpic` yang mengandung micro-expression.

Algoritma spotting **tidak diubah**. Parameter identik dengan
`preprocess-anxiety/process_all_datasets.ipynb`: `top_k_apex=0`, `distance_threshold=1`,
`prominence_threshold=0.005`, `cutoff_ratio=0.3`, `min_phase_duration=2`, `α=1.0`.

## Face alignment

Satu tahap ditambahkan **sebelum** perhitungan magnitude — di luar `ApexPhase` dan
`ApexSmoother`, jadi algoritma spotting sendiri tetap utuh.

Tanpa alignment, kotak ROI dihitung ulang dari landmark setiap frame. Landmark dlib bergetar
sekitar 1 px dan ukuran kotaknya berayun sampai 12 px; karena tiap ROI di-`resize` ke ukuran
tetap, peregangan itu terbaca POC sebagai gerakan. Akibatnya magnitude punya noise floor tinggi
dan tidak pernah mendekati nol walaupun subjek sedang diam.

Dengan `face_alignment=True`, frame valid pertama dijadikan referensi. Frame berikutnya
diregistrasi ke referensi lewat similarity transform (skala + rotasi + translasi) memakai
**landmark rigid saja** — batang hidung, dasar hidung, sudut mata. Mulut dan alis sengaja
dikecualikan karena justru bagian itulah yang bergerak saat ekspresi. ROI kemudian dipotong
dari kotak referensi yang tetap untuk seluruh video.

Rujukan: RMES 2023, Tabel V — face alignment memangkas false positive dari 158 ke 117
dan menaikkan F1 sebesar 21,7% pada CAS(ME)².

Set `face_alignment=False` untuk menghasilkan baseline pembanding. Output kedua varian
disimpan terpisah (`signals/` vs `signals_aligned/`), jadi keduanya bisa hidup berdampingan
untuk tabel ablation.

**Output** (`output/casme2/`)
- `signals_aligned/{subject}/{video}.npy` — magnitude mentah, di-cache; bagian termahal cukup dihitung sekali
- `signals_aligned/{subject}/{video}.xlsx` — raw + smoothed per frame, penanda onset/apex/offset
- `signals_aligned/{subject}/{video}.png` — plot fase
- `dataset_aligned/{subject}/{video}/event_XXXXX-XXXXX/` — frame klip hasil spotting
- `metadata_spotting_aligned.xlsx` — semua interval dalam **nomor img**, siap dipakai `[02]`
- `ground_truth_micro.xlsx` — 57 GT micro + label emosi

In [11]:
import gc
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'casme2':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'casme2') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'casme2'))

from apex import ApexPhase, ApexSmoother, ApexPhaseVisualizer

import casme2_lib as cl

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/ryuko/skripsi/Skripsi/Convat-1st


In [12]:
# face_alignment=False -> perilaku lama, dipakai sebagai baseline ablation
config = cl.Casme2Config(project_root=PROJECT_ROOT, face_alignment=True)

MAX_WORKERS = min(4, os.cpu_count() or 1)
SAVE_CLIPS = True   # simpan frame klip hasil spotting (dibutuhkan notebook [03])

for path in (config.output_root, config.signal_root, config.clip_root, config.eval_root):
    cl.ensure_dir(path)

if not config.rawpic_root.exists():
    raise FileNotFoundError(f'rawpic tidak ditemukan: {config.rawpic_root}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'shape predictor tidak ditemukan: {config.predictor_path}')

print(f'DATASET      = {config.dataset_root}')
print(f'OUTPUT       = {config.output_root}')
print(f'SIGNALS      = {config.signal_root}')
print(f'MAX_WORKERS  = {MAX_WORKERS}')
print(f'FACE_ALIGN   = {config.face_alignment}')
print(f'alpha        = {config.alpha}   (height = mean + alpha * std)')
print(f'top_k_apex   = {config.top_k_apex}, distance = {config.distance_threshold}, '
      f'prominence = {config.prominence_threshold}, cutoff = {config.cutoff_ratio}')
print(f'min_phase_duration = {config.min_phase_duration}')

DATASET      = /home/ryuko/skripsi/Skripsi/Convat-1st/dataset/CAS-ME2
OUTPUT       = /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2
SIGNALS      = /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/signals_aligned
MAX_WORKERS  = 4
FACE_ALIGN   = True
alpha        = 1.0   (height = mean + alpha * std)
top_k_apex   = 0, distance = 1, prominence = 0.005, cutoff = 0.3
min_phase_duration = 2


In [13]:
ground_truth = cl.load_ground_truth(config, micro_only=True)
ground_truth.to_excel(config.ground_truth_path, index=False)

video_keys = cl.micro_video_keys(ground_truth)

unresolved = ground_truth[ground_truth['video_dir'].isna()]
if len(unresolved):
    raise RuntimeError(f'{len(unresolved)} baris GT gagal dipetakan ke folder rawpic')

print(f'GT micro-expression : {len(ground_truth)}')
print(f'Video ber-micro     : {len(video_keys)}')
print(f'Subjek ber-micro    : {ground_truth["subject"].nunique()}')
print()
print(ground_truth['emotion'].value_counts().to_string())
print()
print(f'Ground truth disimpan: {config.ground_truth_path}')
ground_truth[['subject', 'video', 'clip_name', 'onset', 'apex', 'offset', 'duration', 'self_emotion', 'emotion']].head()

GT micro-expression : 57
Video ber-micro     : 33
Subjek ber-micro    : 14

emotion
negative    21
others      21
positive    15

Ground truth disimpan: /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/ground_truth_micro.xlsx


,subject,video,clip_name,onset,apex,offset,duration,self_emotion,emotion
0,s15,15_0102eatingworms,disgust2_3,699,703,707,9,disgust,negative
1,s15,15_0502funnyerrors,happy1_1,138,142,148,11,happiness,positive
2,s16,16_0101disgustingteeth,disgust1_3,552,559,565,14,disgust,negative
3,s16,16_0102eatingworms,disgust2_3,270,275,278,9,disgust,negative
4,s16,16_0502funnyerrors,happy1_1,323,329,334,12,happiness,positive


## Tahap 1 — hitung sinyal magnitude

Bagian paling mahal (~28 ms/frame). Frame dibaca streaming — hanya ROI frame sebelumnya
yang ditahan di memori, jadi video 4.000 frame tetap aman.

Hasilnya di-cache sebagai `.npy`. Menjalankan ulang sel ini akan melewati video yang sudah
selesai, jadi seluruh analisis di hilir bisa diulang tanpa biaya hitung.

In [14]:
WORKER_ALIGNER = None


def init_worker(config_dict: dict) -> None:
    # Satu FaceAligner per proses. Objek dlib tidak bisa di-pickle, jadi harus
    # dibangun di dalam worker, bukan dikirim dari proses induk.
    global WORKER_ALIGNER
    WORKER_ALIGNER = cl.FaceAligner(cl.Casme2Config(**config_dict))


def build_signal(task: dict) -> dict:
    # Sinyal magnitude satu video, streaming per frame.
    video_key = task['video_key']
    cache_path = Path(task['cache_path'])
    if cache_path.exists() and not task['overwrite']:
        return {'video_key': video_key, 'status': 'cached',
                'n': int(np.load(cache_path).size), 'detect_fail': 0}

    frame_paths = cl.load_frame_paths(Path(task['video_dir']))
    if len(frame_paths) < 3:
        return {'video_key': video_key, 'status': 'skip', 'n': len(frame_paths), 'detect_fail': 0}

    # Referensi alignment dipasang ulang untuk tiap video.
    WORKER_ALIGNER.reset()
    block_size = WORKER_ALIGNER.config.block_size

    magnitudes = []
    detect_fail = 0
    prev_rois = None
    started = False

    for frame_path in frame_paths:
        image = cv2.imread(str(frame_path))
        rois = WORKER_ALIGNER.rois(image) if image is not None else None
        if rois is None:
            detect_fail += 1

        if not started:
            # Frame valid pertama hanya memasang baseline, belum menghasilkan magnitude.
            if rois is not None:
                prev_rois = rois
                started = True
            continue

        if rois is None:
            magnitudes.append(0.0)
        else:
            magnitudes.append(cl.roi_magnitude(prev_rois, rois, block_size))
            prev_rois = rois

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, np.asarray(magnitudes, dtype=float))
    return {'video_key': video_key, 'status': 'built', 'n': len(magnitudes),
            'detect_fail': detect_fail}

In [15]:
# Casme2Config dibangun ulang di tiap worker supaya parameternya tidak menyimpang.
worker_config = {
    'project_root': str(PROJECT_ROOT),
    'face_alignment': config.face_alignment,
    'padding_x': config.padding_x,
    'padding_y': config.padding_y,
    'block_size': config.block_size,
    'regions': config.regions,
    'target_size': {k: tuple(v) for k, v in config.target_size.items()},
}

tasks = []
for video_key in video_keys:
    row = ground_truth[ground_truth['video_key'] == video_key].iloc[0]
    tasks.append({
        'video_key': video_key,
        'video_dir': row['video_dir'],
        'cache_path': str(config.signal_root / f'{video_key}.npy'),
        'overwrite': config.overwrite,
    })

start = time.time()
signal_reports = []
with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker,
                         initargs=(worker_config,)) as executor:
    futures = {executor.submit(build_signal, task): task for task in tasks}
    for done, future in enumerate(as_completed(futures), start=1):
        report = future.result()
        signal_reports.append(report)
        print(f"[{done}/{len(tasks)}] {report['status']:>6} {report['video_key']} "
              f"({report['n']} magnitude, gagal deteksi {report['detect_fail']})", flush=True)

print(f'\nSelesai dalam {time.time() - start:.0f} detik')
signal_report_df = pd.DataFrame(signal_reports).sort_values('video_key').reset_index(drop=True)
print(f"Total frame gagal deteksi wajah: {signal_report_df['detect_fail'].sum()}")
signal_report_df

[1/33] cached s15/15_0102eatingworms (1074 magnitude, gagal deteksi 0)
[2/33] cached s16/16_0102eatingworms (1074 magnitude, gagal deteksi 0)
[3/33] cached s16/16_0502funnyerrors (2273 magnitude, gagal deteksi 0)
[4/33] cached s16/16_0505funnyinnovations (2302 magnitude, gagal deteksi 0)
[5/33] cached s16/16_0507climbingthewall (3263 magnitude, gagal deteksi 0)
[6/33] cached s19/19_0402beatingpregnantwoman (4370 magnitude, gagal deteksi 0)
[7/33] cached s23/23_0102eatingworms (1070 magnitude, gagal deteksi 0)
[8/33] cached s24/24_0401girlcrashing (3716 magnitude, gagal deteksi 0)
[9/33] cached s15/15_0502funnyerrors (2272 magnitude, gagal deteksi 0)
[10/33] cached s16/16_0101disgustingteeth (2050 magnitude, gagal deteksi 0)
[11/33] cached s24/24_0507climbingthewall (3264 magnitude, gagal deteksi 0)
[12/33] cached s25/25_0102eatingworms (1074 magnitude, gagal deteksi 0)
[13/33] cached s25/25_0101disgustingteeth (2053 magnitude, gagal deteksi 0)
[14/33] cached s27/27_0101disgustingteeth 

,video_key,status,n,detect_fail
0,s15/15_0102eatingworms,cached,1074,0
1,s15/15_0502funnyerrors,cached,2272,0
2,s16/16_0101disgustingteeth,cached,2050,0
3,s16/16_0102eatingworms,cached,1074,0
4,s16/16_0502funnyerrors,cached,2273,0
5,s16/16_0505funnyinnovations,cached,2302,0
6,s16/16_0507climbingthewall,cached,3263,0
7,s19/19_0402beatingpregnantwoman,cached,4370,0
8,s23/23_0102eatingworms,cached,1070,0
9,s24/24_0401girlcrashing,cached,3716,0


### Cek cepat kualitas sinyal

Kalau alignment bekerja, magnitude pada frame yang subjeknya diam akan turun mendekati nol,
dan autokorelasi lag-1..5 akan menunjukkan struktur temporal. Autokorelasi yang langsung
jatuh ke nol setelah lag-1 menandakan sinyal masih didominasi noise per-frame.

In [16]:
def autocorrelation(signal: np.ndarray, max_lag: int = 5) -> np.ndarray:
    centered = signal - signal.mean()
    denom = np.sum(centered ** 2)
    if denom == 0:
        return np.zeros(max_lag)
    full = np.correlate(centered, centered, 'full')[len(centered) - 1:] / denom
    return full[1:max_lag + 1]


quality_rows = []
for video_key in video_keys:
    cache_path = config.signal_root / f'{video_key}.npy'
    if not cache_path.exists():
        continue
    values = np.load(cache_path)
    if values.size < 10:
        continue
    quality_rows.append({
        'video_key': video_key,
        'n': int(values.size),
        'median': float(np.median(values)),
        'p05': float(np.percentile(values, 5)),
        'frac_below_0.05': float((values < 0.05).mean()),
        'ac_lag1': float(autocorrelation(values)[0]),
        'ac_lag3': float(autocorrelation(values)[2]),
    })

quality = pd.DataFrame(quality_rows)
print(f"noise floor  : median {quality['median'].median():.4f}, p05 {quality['p05'].median():.4f}")
print(f"autokorelasi : lag1 {quality['ac_lag1'].median():.3f}, lag3 {quality['ac_lag3'].median():.3f}")
print(f"frame hampir diam (< 0.05): {quality['frac_below_0.05'].median() * 100:.1f}%")
quality.head(10)

noise floor  : median 0.1866, p05 0.0591
autokorelasi : lag1 0.313, lag3 0.089
frame hampir diam (< 0.05): 3.7%


,video_key,n,median,p05,frac_below_0.05,ac_lag1,ac_lag3
0,s15/15_0102eatingworms,1074,0.129176,0.048654,0.054935,0.313448,0.053483
1,s15/15_0502funnyerrors,2272,0.122191,0.047222,0.066901,0.286693,0.083312
2,s16/16_0101disgustingteeth,2050,0.216084,0.091570,0.003415,0.360619,0.063672
3,s16/16_0102eatingworms,1074,0.207169,0.090039,0.004655,0.451486,0.205775
4,s16/16_0502funnyerrors,2273,0.209160,0.095500,0.004839,0.311949,0.114342
5,s16/16_0505funnyinnovations,2302,0.172728,0.074468,0.008688,0.360620,0.093543
6,s16/16_0507climbingthewall,3263,0.167100,0.063368,0.023904,0.352598,0.083593
7,s19/19_0402beatingpregnantwoman,4370,0.331121,0.083244,0.012815,0.291641,0.022781
8,s23/23_0102eatingworms,1070,1.191030,0.518375,0.000000,0.335864,0.130816
9,s24/24_0401girlcrashing,3716,0.147118,0.040146,0.086383,0.322070,0.127317


## Tahap 2 — deteksi event

Logika `detect_events()` disalin persis dari `process_all_datasets.ipynb`.
Satu-satunya tambahan: hasilnya dikonversi ke **nomor img** supaya sebanding dengan ground truth.

In [17]:
def detect_events(magnitudes: list[float]) -> tuple[list[float], list[dict], dict]:
    # Identik dengan process_all_datasets.ipynb, kecuali alpha dibuat eksplisit (default 1.0).
    smoothed = ApexSmoother.smooth(magnitudes)
    signal_arr = np.asarray(smoothed, dtype=float)
    height_threshold = float(signal_arr.mean() + config.alpha * signal_arr.std())

    detector = ApexPhase(
        distance_threshold=config.distance_threshold,
        prominence_threshold=config.prominence_threshold,
        cutoff_ratio=config.cutoff_ratio,
    )
    apex_indices = detector.find_top_k_apex(smoothed, k=config.top_k_apex, height=height_threshold)
    phases = detector.find_phase(smoothed, apex_indices) if apex_indices else {}

    events = []
    for event_no, apex_idx in enumerate(apex_indices, start=1):
        phase = phases[apex_idx]
        duration = int(phase['end'] - phase['start'])
        if duration < config.min_phase_duration:
            continue
        events.append({
            'event_no': event_no,
            'onset_signal': int(phase['start']),
            'apex_signal': int(apex_idx),
            'offset_signal': int(phase['end']),
            # --- konversi ke nomor img: ini yang dipakai untuk IoU ---
            'onset_img': cl.signal_index_to_img_number(phase['start']),
            'apex_img': cl.signal_index_to_img_number(apex_idx),
            'offset_img': cl.signal_index_to_img_number(phase['end']),
            'duration_signal': duration,
        })

    window_length = ApexSmoother.calculate_window_length(len(magnitudes))
    meta = {
        'window_length': window_length,
        'polyorder': ApexSmoother.calculate_polyorder(window_length),
        'height_threshold': height_threshold,
    }
    return smoothed, events, meta

In [18]:
def save_event_clip(frame_paths: list, out_dir: Path, event: dict, n_frames: int) -> dict:
    # Simpan frame onset..offset. `frame_paths` dikirim dari pemanggil, TIDAK
    # di-listing ulang per event -- iterdir() atas 2000-4000 file, dikali 1400+
    # event, itu bagian paling lambat kalau dibiarkan di dalam fungsi ini.
    onset = max(1, min(event['onset_img'], n_frames))
    offset = max(1, min(event['offset_img'], n_frames))
    apex = max(1, min(event['apex_img'], n_frames))

    cl.ensure_dir(out_dir)
    saved = 0
    for write_idx, img_no in enumerate(range(onset, offset + 1)):
        image = cv2.imread(str(frame_paths[img_no - 1]))
        if image is None:
            continue
        suffix = '-apex.png' if img_no == apex else '.jpg'
        cv2.imwrite(str(out_dir / f'frame_{write_idx:05d}{suffix}'), image)
        saved += 1
    return {'onset': onset, 'apex': apex, 'offset': offset, 'saved_frames': saved}


# Sinyal sudah tersimpan permanen sebagai .npy, jadi tahap ini murah untuk diulang
# KECUALI penulisan klip. Kalau metadata sudah ada dan overwrite=False, klip tidak
# ditulis ulang -- detect_events tetap jalan (hitungan detik) karena signal_store
# dibutuhkan Tahap 3 untuk plot.
metadata_exists = config.spotting_metadata_path.exists() and not config.overwrite
if metadata_exists:
    print(f'Metadata sudah ada, klip tidak ditulis ulang: {config.spotting_metadata_path}')
    print('Set config.overwrite = True kalau memang ingin membangun ulang klipnya.\n')

metadata_rows = []
signal_store = {}

for idx, video_key in enumerate(video_keys, start=1):
    row = ground_truth[ground_truth['video_key'] == video_key].iloc[0]
    video_dir = Path(row['video_dir'])
    cache_path = config.signal_root / f'{video_key}.npy'
    if not cache_path.exists():
        print(f'[{idx}/{len(video_keys)}] lewati {video_key}: sinyal belum ada')
        continue

    magnitudes = np.load(cache_path).tolist()
    smoothed, events, meta = detect_events(magnitudes)
    n_frames = len(magnitudes) + 1
    signal_store[video_key] = {'magnitudes': magnitudes, 'smoothed': smoothed,
                               'events': events, 'meta': meta}

    # Listing frame satu kali per video, dipakai ulang untuk semua event.
    frame_paths = None

    for event in events:
        clip_name = f"event_{event['onset_img']:05d}-{event['offset_img']:05d}"
        clip_dir = config.clip_root / video_key / clip_name
        already = clip_dir.exists() and not config.overwrite

        if SAVE_CLIPS and not already and not metadata_exists:
            if frame_paths is None:
                frame_paths = cl.load_frame_paths(video_dir)
            saved = save_event_clip(frame_paths, clip_dir, event, n_frames)
        else:
            saved = {'onset': event['onset_img'], 'apex': event['apex_img'],
                     'offset': event['offset_img'],
                     'saved_frames': len(list(clip_dir.glob('frame_*'))) if clip_dir.exists() else 0}

        metadata_rows.append({
            'subject': row['subject'],
            'video': row['video'],
            'video_key': video_key,
            'video_code': row['video_code'],
            'event_no': event['event_no'],
            'event_clip': clip_name,
            'onset_img': saved['onset'],
            'apex_img': saved['apex'],
            'offset_img': saved['offset'],
            'duration_img': saved['offset'] - saved['onset'] + 1,
            'saved_frames': saved['saved_frames'],
            'onset_signal': event['onset_signal'],
            'apex_signal': event['apex_signal'],
            'offset_signal': event['offset_signal'],
            'height_threshold': meta['height_threshold'],
            'window_length': meta['window_length'],
            'polyorder': meta['polyorder'],
            'n_frames_video': n_frames,
            'face_alignment': config.face_alignment,
            'clip_dir': str(clip_dir.relative_to(config.output_root)),
        })

    print(f'[{idx}/{len(video_keys)}] {video_key}: {len(events)} interval', flush=True)
    gc.collect()

metadata = pd.DataFrame(metadata_rows)
metadata.to_excel(config.spotting_metadata_path, index=False)
print(f'\nTotal interval terdeteksi (sum Y): {len(metadata)}')
print(f'Rata-rata per video: {len(metadata) / max(len(video_keys), 1):.1f}')
print(f'Metadata disimpan: {config.spotting_metadata_path}')
metadata.head()

Metadata sudah ada, klip tidak ditulis ulang: /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/metadata_spotting_aligned.xlsx
Set config.overwrite = True kalau memang ingin membangun ulang klipnya.

[1/33] s15/15_0102eatingworms: 21 interval
[2/33] s15/15_0502funnyerrors: 36 interval
[3/33] s16/16_0101disgustingteeth: 39 interval
[4/33] s16/16_0102eatingworms: 15 interval
[5/33] s16/16_0502funnyerrors: 39 interval
[6/33] s16/16_0505funnyinnovations: 37 interval
[7/33] s16/16_0507climbingthewall: 44 interval
[8/33] s19/19_0402beatingpregnantwoman: 89 interval
[9/33] s23/23_0102eatingworms: 23 interval
[10/33] s24/24_0401girlcrashing: 67 interval
[11/33] s24/24_0507climbingthewall: 46 interval
[12/33] s25/25_0101disgustingteeth: 39 interval
[13/33] s25/25_0102eatingworms: 19 interval
[14/33] s27/27_0101disgustingteeth: 35 interval
[15/33] s27/27_0102eatingworms: 23 interval
[16/33] s27/27_0401girlcrashing: 67 interval
[17/33] s27/27_0502funnyerrors: 53 interval
[18/33] s27/27_0503unn

,subject,video,video_key,video_code,event_no,event_clip,onset_img,apex_img,offset_img,duration_img,saved_frames,onset_signal,apex_signal,offset_signal,height_threshold,window_length,polyorder,n_frames_video,face_alignment,clip_dir
0,s15,15_0102eatingworms,s15/15_0102eatingworms,0102,1,event_00010-00036,10,17,36,27,27,8,15,34,0.20236,31,4,1075,True,dataset_aligned/s15/15_0102eatingworms/event_0...
1,s15,15_0102eatingworms,s15/15_0102eatingworms,0102,2,event_00056-00083,56,66,83,28,28,54,64,81,0.20236,31,4,1075,True,dataset_aligned/s15/15_0102eatingworms/event_0...
2,s15,15_0102eatingworms,s15/15_0102eatingworms,0102,3,event_00323-00361,323,348,361,39,39,321,346,359,0.20236,31,4,1075,True,dataset_aligned/s15/15_0102eatingworms/event_0...
3,s15,15_0102eatingworms,s15/15_0102eatingworms,0102,4,event_00388-00404,388,397,404,17,17,386,395,402,0.20236,31,4,1075,True,dataset_aligned/s15/15_0102eatingworms/event_0...
4,s15,15_0102eatingworms,s15/15_0102eatingworms,0102,5,event_00405-00433,405,413,433,29,29,403,411,431,0.20236,31,4,1075,True,dataset_aligned/s15/15_0102eatingworms/event_0...


## Tahap 3 — sinyal per frame + plot

In [19]:
visualizer = ApexPhaseVisualizer()
skipped = 0

for video_key, store in signal_store.items():
    signal_path = config.signal_root / f'{video_key}.xlsx'
    plot_path = config.signal_root / f'{video_key}.png'
    if signal_path.exists() and plot_path.exists() and not config.overwrite:
        skipped += 1
        continue

    events = store['events']
    marker_by_index = {}
    for event in events:
        marker_by_index[event['onset_signal']] = ('onset', event['event_no'])
        marker_by_index[event['apex_signal']] = ('apex', event['event_no'])
        marker_by_index[event['offset_signal']] = ('offset', event['event_no'])

    rows = []
    for i, (raw_value, smooth_value) in enumerate(zip(store['magnitudes'], store['smoothed'])):
        marker, event_no = marker_by_index.get(i, ('', None))
        rows.append({
            'video_key': video_key,
            'signal_index': i,
            'img_number': cl.signal_index_to_img_number(i),
            'raw_magnitude': raw_value,
            'smoothed_magnitude': smooth_value,
            'height_threshold': store['meta']['height_threshold'],
            'event_marker': marker,
            'event_no': event_no,
        })

    signal_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_excel(signal_path, index=False)

    if events:
        visualizer.plot_phases(
            signal=store['smoothed'],
            apex_indices=[e['apex_signal'] for e in events],
            phases={e['apex_signal']: {'start': e['onset_signal'], 'end': e['offset_signal']}
                    for e in events},
            title=f'CAS(ME)2 spotting: {video_key}',
            save_path=str(plot_path),
        )

print(f'{skipped} video dilewati (sinyal + plot sudah ada)')
print(f'Sinyal + plot disimpan di: {config.signal_root}')

33 video dilewati (sinyal + plot sudah ada)
Sinyal + plot disimpan di: /home/ryuko/skripsi/Skripsi/Convat-1st/output/casme2/signals_aligned


## Ringkasan

In [20]:
summary = metadata.groupby(['subject', 'video_key']).size().reset_index(name='n_interval')
gt_count = ground_truth.groupby('video_key').size().reset_index(name='n_gt_micro')
summary = summary.merge(gt_count, on='video_key', how='left')
summary['n_gt_micro'] = summary['n_gt_micro'].fillna(0).astype(int)

print(f"Face alignment        : {config.face_alignment}")
print(f"Video diproses        : {summary['video_key'].nunique()}")
print(f"Total interval (sum Y): {int(summary['n_interval'].sum())}")
print(f"Total GT micro (sum X): {int(summary['n_gt_micro'].sum())}")
print(f"Interval per video    : min {summary['n_interval'].min()}, "
      f"median {summary['n_interval'].median():.0f}, max {summary['n_interval'].max()}")
print(f"Lebar interval        : median {metadata['duration_img'].median():.0f} frame "
      f"(GT micro: median {ground_truth['duration'].median():.0f} frame)")
print('\nLanjut ke [02] untuk IoU dan F1 spotting.')
summary

Face alignment        : True
Video diproses        : 33
Total interval (sum Y): 1434
Total GT micro (sum X): 57
Interval per video    : min 11, median 39, max 89
Lebar interval        : median 18 frame (GT micro: median 14 frame)

Lanjut ke [02] untuk IoU dan F1 spotting.


,subject,video_key,n_interval,n_gt_micro
0,s15,s15/15_0102eatingworms,21,1
1,s15,s15/15_0502funnyerrors,36,1
2,s16,s16/16_0101disgustingteeth,39,1
3,s16,s16/16_0102eatingworms,15,1
4,s16,s16/16_0502funnyerrors,39,1
5,s16,s16/16_0505funnyinnovations,37,3
6,s16,s16/16_0507climbingthewall,44,2
7,s19,s19/19_0402beatingpregnantwoman,89,1
8,s23,s23/23_0102eatingworms,23,3
9,s24,s24/24_0401girlcrashing,67,5
